# 02 · Baseline and review

**Question:** Does rule/example context help, and what breaks when the rule changes?

These CPU models are reference baselines, not the final advanced models. `comment_only` learns TF-IDF lexical patterns. `rule_examples` adds comment-to-rule/example cosine similarities and positive-minus-negative similarity features. The latter is predeclared for the starter submission. No test labels influence model choice.

In [ ]:
import os
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML, FileLink
from jigsaw_rules.data import load_data, audit
from jigsaw_rules.runtime import Progress, environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = ["#087f8c", "#bd633b", "#334ea0", "#70923b"]
display(HTML("<div style='padding:18px;background:#edf6f5;border-left:5px solid #087f8c'>"
             "<b>Jigsaw research workspace</b><br>Every result must identify its data and validation protocol.</div>"))
print("Project:", root)


## Run or resume the experiment
Completed stages are reused only when data, source, configuration, package versions, and output checksums match. If a fold is interrupted, that fold restarts; earlier completed folds remain valid. With cloud backup enabled, every completed stage is copied to S3. GPU optimizer-state recovery belongs to the later neural training phase.

In [ ]:
from jigsaw_rules.pipeline import run_baseline
use_cloud = os.environ.get('JIGSAW_CLOUD', '1') == '1'
config_path = root / 'configs/local.json'
if use_cloud and not config_path.exists():
    raise FileNotFoundError('Create configs/local.json from the example, or explicitly set JIGSAW_CLOUD=0 for local verification.')
cloud = json.loads(config_path.read_text()) if use_cloud else None
run_dir = run_baseline(root, cloud=cloud)

## Compare ranking and probability quality
A strong seen-rule result with weak held-out-rule AUC indicates poor policy transfer. AUC does not establish calibration. Keep the held-out-rule analysis separate from leaderboard results.

In [ ]:
results = json.loads((run_dir / 'review/results.json').read_text())
summary = pd.DataFrame([{'Model': r['model'], 'Protocol': r['protocol'], **{k: v for k, v in r['metrics'].items() if isinstance(v, float)}} for r in results])
display(summary.round(4))
fig = px.bar(summary, x='Protocol', y='rule_macro_auc', color='Model', barmode='group', title='Baseline validation comparison')
fig.update_yaxes(range=[0, 1])
fig.show()
display(FileLink(str(run_dir / 'review/report.html')))

## Investigate failures
Review errors by rule and subreddit locally. Row IDs below let you join to private comments when needed. Avoid publishing raw comments or model outputs that expose them without a deliberate review.

In [ ]:
errors = pd.read_csv(run_dir / 'review/error_review.csv')
display(errors.head(12))
coefficients = pd.read_csv(run_dir / 'full_training/coefficients.csv')
context = coefficients[coefficients.feature.str.contains('similarity')]
display(context)
print('Preview submission:', run_dir / 'full_training/submission.csv')

## Next phase
Compare an embedding/example matcher and a cross-encoder, then an instruction model with LoRA. Add confidence intervals, stricter near-duplicate audits, rule/example ablations, and nested calibration before selecting a final ensemble. The baseline's lexical coefficients describe association, not a causal explanation.

Use **kaggle/submission.ipynb** for portable offline inference. It regenerates the submission using whichever test rows Kaggle provides. Late scoring remains dependent on your signed-in account's eligibility.